In [2]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# Cart-pole dynamics
def cartpole_dynamics(state, u, params):
    x, x_dot, theta, theta_dot = state
    mc, mp, l, g = params

    sin_theta = np.sin(theta)
    cos_theta = np.cos(theta)

    theta_ddot = (
        g * sin_theta + cos_theta * (-mp * l * theta_dot**2 * sin_theta - u) / (mc + mp)
    ) / (l * (4/3 - mp * cos_theta**2 / (mc + mp)))

    x_ddot = (
        mp * l * (theta_dot**2 * sin_theta - theta_ddot * cos_theta) + u
    ) / (mc + mp)

    return np.array([x_dot, x_ddot, theta_dot, theta_ddot])

# Simulate using linear policy u = -Kx
def simulate_policy(initial_state, K, params, Q, R, dt, sim_steps, u_bounds):
    state = np.array(initial_state)
    trajectory = [state]
    controls = []
    total_cost = 0
    for _ in range(sim_steps):
        u = (-K @ state).item()
        u = np.clip(u, *u_bounds)
        controls.append(u)
        total_cost += state.T @ Q @ state + u * R * u
        state_dot = cartpole_dynamics(state, u, params)
        state = state + dt * state_dot
        trajectory.append(state)
    total_cost += state.T @ Q @ state  # terminal cost
    return np.array(trajectory), np.array(controls).flatten(), total_cost

# Train a linear policy using random search (simple policy search)
def train_policy_model_based(initial_state, params, Q, R, dt, sim_steps, u_bounds, iterations=100, alpha=0.1):
    d = len(initial_state)
    K_best = np.zeros((1, d))
    best_cost = float('inf')

    for _ in range(iterations):
        K_candidate = K_best + alpha * np.random.randn(1, d)
        _, _, cost = simulate_policy(initial_state, K_candidate, params, Q, R, dt, sim_steps, u_bounds)
        if cost < best_cost:
            K_best = K_candidate
            best_cost = cost
    return K_best

# --- Load data
trajectories = np.load('fractional_system_trajectories.npy')
U_optimal = np.load('optimal_control_U.npy')
Q_matrices = np.load('LQR_Q.npy')
R_matrices = np.load('LQR_R.npy')

# --- System settings
mc, mp, l, g = 1.0, 0.1, 1.0, 9.81
params = (mc, mp, l, g)
dt = 0.1
sim_time = 3.2
sim_steps = int(sim_time / dt)
u_bounds = (-0.5, 0.5)

maes, mses = [], []
start_time = time.time()

# --- Training loop
for i in range(8000, 8050):
    x0 = trajectories[i, 0]
    Q = Q_matrices[i]
    R = R_matrices[i]

    # Train policy
    K = train_policy_model_based(x0, params, Q, R, dt, sim_steps, u_bounds, iterations=100, alpha=0.1)

    # Evaluate policy
    traj, u_pred, _ = simulate_policy(x0, K, params, Q, R, dt, sim_steps, u_bounds)

    mae = mean_absolute_error(u_pred, U_optimal[i])
    mse = mean_squared_error(u_pred, U_optimal[i])
    print(f"Iteration {i}, MAE: {mae:.4f}, MSE: {mse:.4f}")
    maes.append(mae)
    mses.append(mse)

end_time = time.time()
print("Total runtime: {:.2f} seconds".format(end_time - start_time))

Iteration 8000, MAE: 0.1625, MSE: 0.0723
Iteration 8001, MAE: 0.3616, MSE: 0.1668
Iteration 8002, MAE: 0.2993, MSE: 0.1239
Iteration 8003, MAE: 0.4295, MSE: 0.2096
Iteration 8004, MAE: 0.4514, MSE: 0.2249
Iteration 8005, MAE: 0.2238, MSE: 0.0653
Iteration 8006, MAE: 0.3885, MSE: 0.1689
Iteration 8007, MAE: 0.4123, MSE: 0.2023
Iteration 8008, MAE: 0.4638, MSE: 0.2189
Iteration 8009, MAE: 0.5430, MSE: 0.2992
Iteration 8010, MAE: 0.4306, MSE: 0.2051
Iteration 8011, MAE: 0.5219, MSE: 0.2747
Iteration 8012, MAE: 0.3177, MSE: 0.1298
Iteration 8013, MAE: 0.5406, MSE: 0.2966
Iteration 8014, MAE: 0.5365, MSE: 0.2919
Iteration 8015, MAE: 0.5224, MSE: 0.2867
Iteration 8016, MAE: 0.4264, MSE: 0.2024
Iteration 8017, MAE: 0.4694, MSE: 0.2234
Iteration 8018, MAE: 0.3715, MSE: 0.1579
Iteration 8019, MAE: 0.3785, MSE: 0.1724
Iteration 8020, MAE: 0.4123, MSE: 0.1989
Iteration 8021, MAE: 0.4325, MSE: 0.2094
Iteration 8022, MAE: 0.4115, MSE: 0.1909
Iteration 8023, MAE: 0.4171, MSE: 0.1951
Iteration 8024, 